# NutriLens: huấn luyện model nhận dạng món ăn Việt

Notebook này huấn luyện lại model AI cho website NutriLens, **chỉ nhận dạng món Việt**, dùng bộ dữ liệu
[30VNFoods](https://www.kaggle.com/datasets/quandang/vietnamese-foods) (30 món).

**Khác với model cũ (CNN tự xây, 41,6% trên 20 món Food-101):**

| | Model cũ | Model mới |
|---|---|---|
| Kiến trúc | 3 lớp Conv tự huấn luyện từ đầu | **EfficientNetV2-B0** học chuyển giao (transfer learning) từ ImageNet |
| Kích thước ảnh | 128×128 | 224×224 |
| Huấn luyện | 1 giai đoạn | 2 giai đoạn: huấn luyện phần đầu phân loại, rồi tinh chỉnh toàn mạng |
| Dữ liệu | Dùng nguyên | Loại ảnh hỏng, **ảnh trùng lặp** và ảnh trùng mang 2 nhãn khác nhau; chia lại 70/15/15 theo từng lớp để tập test không lẫn ảnh của tập train |
| Mất cân bằng lớp | Không xử lý | Trọng số lớp (class weight) |
| Chống quá khớp | L2 + Dropout | Tăng cường ảnh, label smoothing, Dropout, dừng sớm |
| Đánh giá | Accuracy | Accuracy, **top-3**, F1 từng lớp, ma trận nhầm lẫn |
| Xuất ra | `.h5` | `food_model.tflite` (float16, nhẹ) + `class_names.json`, chạy thẳng trên web |

**Cách chạy:** `Runtime → Change runtime type → T4 GPU`, rồi `Runtime → Run all`. Mất khoảng 30–60 phút.
Kết quả được lưu vào Google Drive, thư mục `KY_THUAT_HOC_SAU/vn_food`, nên mất kết nối giữa chừng vẫn còn checkpoint.

## 1. Thư viện và cấu hình

In [ ]:
import os, sys, json, re, random, hashlib, unicodedata, time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

IN_COLAB = 'google.colab' in sys.modules
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

GPUS = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPU:', GPUS or 'KHÔNG CÓ (hãy bật T4 GPU)')

In [ ]:
# ── Đường dẫn ───────────────────────────────────────────────
OUT_DIR   = Path(os.environ.get('VNFOOD_OUT_DIR', '/content/drive/MyDrive/KY_THUAT_HOC_SAU/vn_food'))
DATA_DIR  = Path(os.environ['VNFOOD_DATA_DIR']) if os.environ.get('VNFOOD_DATA_DIR') else None  # None: tải từ Kaggle
CACHE_DIR = Path(os.environ.get('VNFOOD_CACHE_DIR', '/content/vnfood_cache'))  # ảnh đã làm sạch, ổ cục bộ cho nhanh

# ── Tham số ─────────────────────────────────────────────────
IMG_SIZE      = int(os.environ.get('VNFOOD_IMG_SIZE', 224))   # kích thước đầu vào model
CACHE_SIZE    = int(IMG_SIZE * 256 / 224)                     # ảnh lưu sẵn lớn hơn một chút để cắt ngẫu nhiên
BATCH_SIZE    = 32
EPOCHS_HEAD   = int(os.environ.get('VNFOOD_EPOCHS_HEAD', 6))   # giai đoạn 1: chỉ huấn luyện phần phân loại
EPOCHS_FINE   = int(os.environ.get('VNFOOD_EPOCHS_FINE', 20))  # giai đoạn 2: tinh chỉnh toàn mạng
LR_HEAD, LR_FINE = 1e-3, 1e-4
LABEL_SMOOTHING  = 0.1
MIN_PER_CLASS    = int(os.environ.get('VNFOOD_MIN_PER_CLASS', 40))  # lớp ít ảnh hơn sẽ bị bỏ
WEIGHTS          = os.environ.get('VNFOOD_WEIGHTS', 'imagenet')

# Tên thư mục trong dataset -> slug món ăn trên website (để web tự gắn đúng món)
ALIASES = {'canh_chua': 'canh_chua_ca'}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Kết quả lưu tại:', OUT_DIR)

## 2. Tải bộ dữ liệu 30VNFoods

Dùng `kagglehub`. Nếu báo lỗi cần đăng nhập: vào kaggle.com → Settings → **Create New Token** để tải `kaggle.json`,
rồi trong Colab bấm biểu tượng 🔑 (Secrets) bên trái, thêm `KAGGLE_USERNAME` và `KAGGLE_KEY` với giá trị trong file đó, bật quyền truy cập cho notebook và chạy lại cell.

In [ ]:
if DATA_DIR is None:
    try:
        import kagglehub
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True)
        import kagglehub
    DATA_DIR = Path(kagglehub.dataset_download('quandang/vietnamese-foods'))
print('Dữ liệu gốc:', DATA_DIR)
print('Cấu trúc thư mục (2 cấp đầu):')
for p in sorted(DATA_DIR.iterdir()):
    sub = [c.name for c in sorted(p.iterdir())][:6] if p.is_dir() else []
    print(' ', p.name, '->', sub, '...' if p.is_dir() and len(list(p.iterdir())) > 6 else '')

## 3. Làm sạch dữ liệu

- Nhãn lấy từ tên thư mục chứa ảnh, đổi thành slug không dấu (`Bún bò Huế` → `bun_bo_hue`), bỏ qua các thư mục tên `Train`/`Test`/`Validate`.
- **Gộp tất cả tập có sẵn rồi chia lại**, vì bộ dữ liệu gốc có ảnh trùng giữa train và test (làm điểm test cao ảo).
- Ảnh trùng hệt nhau (cùng mã băm MD5): giữ 1 ảnh. Ảnh trùng mà **mang 2 nhãn khác nhau**: bỏ hết vì không biết nhãn nào đúng.
- Ảnh không mở được: bỏ. Ảnh còn lại được xoay đúng chiều (EXIF), đổi sang RGB và lưu sẵn kích thước nhỏ để huấn luyện nhanh.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.jfif'}
SPLIT_NAMES = {'train', 'training', 'test', 'testing', 'val', 'valid', 'validate', 'validation',
               'images', 'image', 'data', 'dataset'}

def slugify(name: str) -> str:
    s = unicodedata.normalize('NFKD', name.replace('đ', 'd').replace('Đ', 'D'))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')
    return ALIASES.get(s, s)

def label_of(path: Path):
    for part in reversed(path.parent.relative_to(DATA_DIR).parts):
        if part.lower() not in SPLIT_NAMES and not part.startswith('.'):
            return slugify(part)
    return None

files = [p for p in DATA_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT]
print(f'Tìm thấy {len(files):,} file ảnh')

def md5(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

with ThreadPoolExecutor(8) as ex:
    hashes = list(ex.map(md5, files))

labels_by_hash = defaultdict(set)
first_file = {}
for f, h in zip(files, hashes, strict=True):
    lab = label_of(f)
    if lab:
        labels_by_hash[h].add(lab)
        first_file.setdefault(h, f)

conflict = {h for h, labs in labels_by_hash.items() if len(labs) > 1}
unique = {h: next(iter(labs)) for h, labs in labels_by_hash.items() if h not in conflict}
print(f'Ảnh khác nhau: {len(labels_by_hash):,} | trùng lặp đã gộp: {len(files) - len(labels_by_hash):,} '
      f'| trùng nhưng khác nhãn (bỏ): {len(conflict):,}')

In [ ]:
def prepare(item):
    # Mở, xoay đúng chiều, đổi RGB, thu nhỏ rồi lưu JPEG vào CACHE_DIR/<nhãn>/<md5>.jpg
    h, lab = item
    dst = CACHE_DIR / lab / f'{h}.jpg'
    if dst.exists():
        return h, lab, True
    try:
        with Image.open(first_file[h]) as im:
            im = ImageOps.exif_transpose(im).convert('RGB')
            if min(im.size) < 32:
                return h, lab, False
            im = im.resize((CACHE_SIZE, CACHE_SIZE), Image.Resampling.BICUBIC)
            dst.parent.mkdir(parents=True, exist_ok=True)
            im.save(dst, 'JPEG', quality=95)
        return h, lab, True
    except Exception:
        return h, lab, False

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    results = list(ex.map(prepare, unique.items()))
ok = [(h, lab) for h, lab, good in results if good]
print(f'Ảnh hợp lệ: {len(ok):,} | ảnh hỏng (bỏ): {len(results) - len(ok):,} | {time.time() - t0:.0f}s')

counts = Counter(lab for _, lab in ok)
dropped = sorted(c for c, n in counts.items() if n < MIN_PER_CLASS)
if dropped:
    print('Bỏ lớp quá ít ảnh:', dropped)
CLASS_NAMES = sorted(c for c, n in counts.items() if n >= MIN_PER_CLASS)
NUM_CLASSES = len(CLASS_NAMES)
paths  = [str(CACHE_DIR / lab / f'{h}.jpg') for h, lab in ok if lab in CLASS_NAMES]
labels = [CLASS_NAMES.index(lab) for h, lab in ok if lab in CLASS_NAMES]

print(f'\n{NUM_CLASSES} lớp, {len(paths):,} ảnh')
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(CLASS_NAMES, [counts[c] for c in CLASS_NAMES], color='#2f855a')
ax.set_title('Số ảnh mỗi món sau khi làm sạch'); ax.tick_params(axis='x', rotation=75)
plt.tight_layout(); plt.savefig(OUT_DIR / 'class_distribution.png', dpi=130); plt.show()

## 4. Chia train / validation / test (70 / 15 / 15, giữ tỉ lệ từng lớp)

In [ ]:
p_train, p_rest, y_train, y_rest = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(p_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=SEED)
print(f'Train {len(p_train):,} | Val {len(p_val):,} | Test {len(p_test):,}')

# Trọng số lớp: món ít ảnh được tính nặng hơn khi huấn luyện
n_per_class = np.bincount(y_train, minlength=NUM_CLASSES)
CLASS_WEIGHT = {i: len(y_train) / (NUM_CLASSES * n) for i, n in enumerate(n_per_class)}
print(f'Trọng số lớp từ {min(CLASS_WEIGHT.values()):.2f} đến {max(CLASS_WEIGHT.values()):.2f}')

## 5. Pipeline dữ liệu

Đầu vào model là ảnh RGB chia cho 255 (giá trị 0–1), kéo về 224×224, **giống hệt cách website xử lý ảnh** người dùng tải lên.
Tập train được tăng cường: cắt ngẫu nhiên, lật ngang, xoay, phóng to/thu nhỏ, đổi độ sáng và độ tương phản.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom((-0.15, 0.1)),
    layers.RandomTranslation(0.06, 0.06),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15, value_range=(0.0, 1.0)),
], name='augment')

def decode(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.cast(img, tf.float32) / 255.0

def make_ds(p, y, training):
    ds = tf.data.Dataset.from_tensor_slices((p, tf.one_hot(y, NUM_CLASSES)))
    if training:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    def load(path, label):
        img = decode(path)
        if training:
            img = tf.image.random_crop(img, (IMG_SIZE, IMG_SIZE, 3))
        else:
            img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method='bicubic')
        return tf.clip_by_value(img, 0.0, 1.0), label
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0.0, 1.0), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(p_train, y_train, True)
val_ds   = make_ds(p_val, y_val, False)
test_ds  = make_ds(p_test, y_test, False)

xb, yb = next(iter(train_ds))
plt.figure(figsize=(12, 5))
for i in range(min(10, len(xb))):
    plt.subplot(2, 5, i + 1); plt.imshow(xb[i].numpy()); plt.axis('off')
    plt.title(CLASS_NAMES[int(np.argmax(yb[i]))], fontsize=9)
plt.suptitle('Ảnh train sau tăng cường'); plt.tight_layout(); plt.show()

## 6. Mô hình: EfficientNetV2-B0 + học chuyển giao

- Phần trích đặc trưng (backbone) đã học trên 1,2 triệu ảnh ImageNet, nên nhận ra cạnh, kết cấu, hình khối tốt hơn nhiều so với CNN tự huấn luyện trên vài nghìn ảnh.
- Lớp `Rescaling(255)` đưa ảnh 0–1 về 0–255 như EfficientNetV2 yêu cầu, nhờ vậy website không phải đổi cách xử lý ảnh.
- Backbone luôn chạy BatchNorm ở chế độ suy luận (`training=False`) để giai đoạn tinh chỉnh không làm hỏng thống kê đã học.

In [ ]:
if GPUS:
    keras.mixed_precision.set_global_policy('mixed_float16')  # nhanh gấp ~2 lần trên GPU T4

def build_model(weights=WEIGHTS):
    base = keras.applications.EfficientNetV2B0(include_top=False, weights=weights,
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')        # RGB 0-1, giống website
    x = layers.Rescaling(255.0, name='to_0_255')(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3, name='dropout')(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='output')(x)
    return keras.Model(inp, out, name='VNFood_EfficientNetV2B0'), base

model, base = build_model()
print(f'Tổng tham số: {model.count_params():,}')

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(3, name='top3')])

## 7. Giai đoạn 1: chỉ huấn luyện phần phân loại (backbone đóng băng)

In [ ]:
CKPT = OUT_DIR / 'best.weights.h5'
def callbacks(patience):
    return [
        keras.callbacks.ModelCheckpoint(CKPT, monitor='val_accuracy', save_best_only=True, save_weights_only=True),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ]

base.trainable = False
compile_model(LR_HEAD)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
               class_weight=CLASS_WEIGHT, callbacks=callbacks(3))

## 8. Giai đoạn 2: tinh chỉnh toàn mạng với tốc độ học nhỏ

In [ ]:
base.trainable = True
for l in base.layers:                       # giữ nguyên BatchNorm
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
compile_model(LR_FINE)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD + EPOCHS_FINE, initial_epoch=len(h1.epoch),
               class_weight=CLASS_WEIGHT, callbacks=callbacks(5))
if CKPT.exists():
    model.load_weights(CKPT)                # trọng số có val_accuracy cao nhất

In [ ]:
hist = {k: h1.history[k] + h2.history[k] for k in ('loss', 'val_loss', 'accuracy', 'val_accuracy')}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for i, (k, title) in enumerate([('loss', 'Loss'), ('accuracy', 'Accuracy')]):
    ax[i].plot(hist[k], label='Train', color='#E24B4A', lw=2)
    ax[i].plot(hist['val_' + k], label='Validation', color='#378ADD', lw=2, ls='--')
    ax[i].axvline(len(h1.epoch) - 0.5, color='gray', ls=':', label='Bắt đầu tinh chỉnh')
    ax[i].set_title(title + ' theo epoch'); ax[i].set_xlabel('Epoch'); ax[i].grid(alpha=.3); ax[i].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / 'training_curves.png', dpi=150); plt.show()

## 9. Đánh giá trên tập test (ảnh model chưa từng thấy)

In [ ]:
y_proba = model.predict(test_ds, verbose=1)
y_pred = y_proba.argmax(1)
y_true = np.array(y_test)
acc  = float((y_pred == y_true).mean())
top3 = float(np.mean([t in np.argsort(p)[-3:] for t, p in zip(y_true, y_proba, strict=True)]))
f1m  = float(f1_score(y_true, y_pred, average='macro'))
print('=' * 50)
print(f'  Accuracy (top-1): {acc*100:.2f}%')
print(f'  Top-3 accuracy  : {top3*100:.2f}%')
print(f'  Macro F1        : {f1m:.4f}')
print('=' * 50)
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(16, 13))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                    color='white' if cm_norm[i, j] > .5 else 'black')
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật'); ax.set_title('Ma trận nhầm lẫn trên tập test (tỉ lệ theo hàng)')
fig.colorbar(im, fraction=.03); plt.tight_layout(); plt.savefig(OUT_DIR / 'confusion_matrix.png', dpi=150); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print('Các cặp hay nhầm nhất:')
for idx in np.argsort(off.ravel())[::-1][:8]:
    i, j = divmod(idx, NUM_CLASSES)
    if off[i, j]:
        print(f'  {CLASS_NAMES[i]:>18} -> {CLASS_NAMES[j]:<18} {off[i, j]} ảnh')

## 10. Xuất model cho website

- `food_model.tflite`: trọng số float16 (nhẹ bằng một nửa), chạy bằng LiteRT trên máy chủ 512 MB RAM.
- `class_names.json`: danh sách slug món ăn theo đúng thứ tự đầu ra của model.
- Kiểm tra lại: bản `.tflite` phải cho cùng kết quả với model Keras trên ảnh test.

In [ ]:
keras.mixed_precision.set_global_policy('float32')          # bản xuất chạy float32 trên CPU
export_model, _ = build_model(weights=None)
export_model.set_weights(model.get_weights())
export_model.save(OUT_DIR / 'food_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_path = OUT_DIR / 'food_model.tflite'
tflite_path.write_bytes(converter.convert())
(OUT_DIR / 'class_names.json').write_text(json.dumps(CLASS_NAMES, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'food_model.tflite: {tflite_path.stat().st_size / 1e6:.1f} MB')

# So sánh TFLite với Keras trên tối đa 300 ảnh test
interp = tf.lite.Interpreter(model_path=str(tflite_path))
inp_i, out_i = interp.get_input_details()[0]['index'], interp.get_output_details()[0]['index']
interp.allocate_tensors()
n_check = min(300, len(p_test))
xs = np.concatenate([x.numpy() for x, _ in test_ds.take(int(np.ceil(n_check / BATCH_SIZE)))])[:n_check]
lite = []
for x in xs:
    interp.set_tensor(inp_i, x[None].astype(np.float32)); interp.invoke()
    lite.append(interp.get_tensor(out_i)[0])
lite = np.array(lite)
agree = float((lite.argmax(1) == y_proba[:n_check].argmax(1)).mean())
lite_acc = float((lite.argmax(1) == y_true[:n_check]).mean())
print(f'TFLite khớp Keras: {agree*100:.1f}% | accuracy TFLite trên {n_check} ảnh: {lite_acc*100:.2f}%')

metrics = {'model': 'EfficientNetV2B0', 'img_size': IMG_SIZE, 'classes': NUM_CLASSES,
           'train': len(p_train), 'val': len(p_val), 'test': len(p_test),
           'test_accuracy': round(acc, 4), 'test_top3': round(top3, 4), 'test_macro_f1': round(f1m, 4),
           'tflite_agreement': round(agree, 4), 'epochs_run': len(hist['loss'])}
(OUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

## 11. Tải file về máy

Cần 2 file: **`food_model.tflite`** và **`class_names.json`** (cũng đã nằm trong Drive, thư mục `KY_THUAT_HOC_SAU/vn_food`).
Chép vào thư mục gốc project NutriLens, thay file cũ. `metrics.json` và các ảnh `.png` dùng cho báo cáo.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for name in ('food_model.tflite', 'class_names.json', 'metrics.json'):
        files.download(str(OUT_DIR / name))